# Scriptoria Interactive Notebook: Checkpoints 0 & 1
### Plumbing, Cloud Lake, Contracts & Parser AST Deep Dive

This notebook documents how **Checkpoint 0 (Plumbing & Storage Lake)** and **Checkpoint 1 (Parser Inspector & AST Splitter)** work under the hood in Scriptoria.

**Key Topics Covered:**
1. **Infrastructure Health**: Cloudflare R2 bucket & Supabase `books_catalog` interaction.
2. **Pydantic v1.0 Contract Verification**: Testing canonical schemas & simulated fault rejection (EC-0.1).
3. **Token-Bounded AST Parser**: Splitting novel text into deterministic chapters, scenes, and chunks.
4. **Manifest Classification**: Why classifying sections as `STORY` vs `FRONTMATTER` controls GPU cluster fan-out.

## 1. Setup & Environment Verification

In [ ]:
import sys, os, json, re
from pathlib import Path
from dotenv import load_dotenv

# Add repo root to sys.path
repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

load_dotenv(repo_root / '.env')
print('Repo Root:', repo_root)
print('R2 Bucket:', os.getenv('R2_BUCKET_NAME', 'not configured'))
print('Supabase URL:', os.getenv('SUPABASE_URL', 'not configured'))

## 2. Checkpoint 0: Cloudflare R2 Storage Lake & Supabase Catalog
Query the books catalog and examine the R2 tree layout for the active novel.

In [ ]:
from backend.scripts.pipeline_bridge import list_books, get_r2_tree

catalog = list_books()
books = catalog.get('books', [])
print(f'Found {len(books)} books registered in Supabase catalog:')
for b in books:
    print(f"  - [{b.get('status')}] {b.get('title')} (ID: {b.get('book_id')})")

if books:
    active_id = books[0]['book_id']
    print(f"\nFetching R2 Object Tree for Active Book: {books[0]['title']}...")
    tree = get_r2_tree(active_id)
    print(json.dumps(tree, indent=2))

## 3. Checkpoint 0: Schema Guard & Fault Injection (EC-0.1)
Scriptoria rejects corrupt LLM outputs at the API gateway before they reach downstream stores.

In [ ]:
from backend.scripts.pipeline_bridge import validate_contract_json

# 1. Pristine canonical observation
valid_sample = {
    'schema_version': '1.0.0',
    'book_id': 'demo-book-id',
    'chapter_index': 1,
    'chapter_title': 'Chapter 1: The Mountain Pass',
    'raw_text_r2_key': 'parsed/demo/ch_001.md',
    'scenes': [{
        'scene_id': 'ch001_sc01',
        'scene_order_in_chapter': 1,
        'start_char_offset': 0,
        'end_char_offset': 240,
        'verbatim_excerpt_summary': 'The boy watches the solitary lantern.',
        'setting': {'location_id': 'loc_gate', 'location_name': 'Mountain Gate'},
        'characters_present': [{'entity_id': 'char_marcus', 'name_in_scene': 'Marcus'}],
        'key_visual_beats': [{'beat_order': 1, 'shot_suggestion': 'Wide shot', 'focal_elements': ['lantern'], 'quote_anchor': 'Wait for the signal.'}]
    }],
    'discovered_entities': [{'entity_id': 'char_marcus', 'canonical_name': 'Marcus', 'entity_type': 'CHARACTER'}]
}

print('Testing valid sample:')
res_valid = validate_contract_json(json.dumps(valid_sample))
print('Status:', 'VALID' if res_valid['valid'] else 'REJECTED')

# 2. Injected corrupt sample (Hallucinated key forbidden by Pydantic extra='forbid')
corrupt_sample = dict(valid_sample)
corrupt_sample['hallucinated_unauthorized_key'] = 'MODEL_LEAK_ATTEMPT'

print('\nTesting corrupt sample with unauthorized key:')
res_corrupt = validate_contract_json(json.dumps(corrupt_sample))
print('Status:', 'VALID' if res_corrupt['valid'] else 'REJECTED')
print('Errors caught:', res_corrupt.get('errors'))

## 4. Checkpoint 1: Parser Inspector & AST Decomposition
Transforming raw text into AST nodes and token-budgeted chunks with exact character spans.

In [ ]:
SAMPLE_TEXT = """# Chapter 1: The Mountain Pass
The cold wind howled across the frozen ramparts. Marcus stood motionless, waiting.

<SCENE_BREAK />

Three hours later, the lantern flared twice in rapid succession. The riders emerged from the pines.
"""

# AST Decomposition Logic
scenes = [s.strip() for s in re.split(r'<SCENE_BREAK\s*/>', SAMPLE_TEXT) if s.strip()]
print(f'Detected {len(scenes)} scenes in chapter:')
for idx, sc in enumerate(scenes, start=1):
    print(f'  [Scene {idx}] Words: {len(sc.split())} | Length: {len(sc)} chars')
    print(f'    Preview: "{sc[:60]}..."')

## 5. Downstream Impact: Manifest Chapter Filtering
Checkpoint 2 (Kaggle Swarm Extraction) filters out all frontmatter to avoid burning GPU inference quota on copyright pages.

In [ ]:
manifest_mock = [
    {'chapter_index': 0, 'title': 'Cover & Copyright', 'classification': 'FRONTMATTER'},
    {'chapter_index': 1, 'title': 'Dedication', 'classification': 'FRONTMATTER'},
    {'chapter_index': 2, 'title': 'Chapter 1: The Boy', 'classification': 'STORY'},
    {'chapter_index': 3, 'title': 'Chapter 2: The Merchant', 'classification': 'STORY'},
    {'chapter_index': 4, 'title': 'About the Author', 'classification': 'BACKMATTER'},
]

queued_for_gpu_swarm = [ch for ch in manifest_mock if ch['classification'] == 'STORY']
print(f'Total book sections: {len(manifest_mock)}')
print(f'Queued for Kaggle Dual-T4 Swarm: {len(queued_for_gpu_swarm)} chapters')
for ch in queued_for_gpu_swarm:
    print(f"  -> Section {ch['chapter_index']}: {ch['title']}")